# TwoTank - Stage 4c - Sparse Symbolic vs NN Autograd Safety Gradients

Compare sparse symbolic and neural autograd safety gradients.


In [ ]:
import sys, warnings, time, statistics, copy
from pathlib import Path
warnings.filterwarnings("ignore")

_p = Path.cwd()
while not (_p / "src" / "sdpc").exists():
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

import torch
try:
    import pandas as pd
except Exception:
    pd = None

from sdpc.config import load_config
from sdpc.registry import make_system
from sdpc.sindy import load_model
from sdpc.training import build_nn_policy
from sdpc.adaptation import SymbolicSafetyJacobian
from sdpc.adaptation.analytic import sindy_step
from sdpc.adaptation.rollout import current_reference, hold_current_reference, predict_rollout
from sdpc.adaptation.barrier import (
    rollout_barrier_loss,
    rollout_box_relu_loss,
    rollout_box_barrier_loss,
)

SYSTEM = "twotank"
device = torch.device("cpu")
system = make_system(SYSTEM, device=device)
CONFIGS = Path.cwd().parent / "configs"
RESULTS = Path.cwd().parent / "results"

print(f"System: {SYSTEM}, device={device}")

In [ ]:
cfg = load_config(CONFIGS / "safe.yaml")
from sdpc.eval import sample_scenario
safe_cfg = cfg.get("safe", {})
action_scale = safe_cfg.get("action_scale", cfg.get("action_scale", 1.0))
barrier_eps = safe_cfg.get("barrier_eps", 1e-6)
safety_loss_tol = safe_cfg.get("safety_loss_tol", 1e-10)

spec = system.safety_specs(cfg)
sindy = system.perturbed_sindy_model(cfg)
benchmark_data = sample_scenario(system, cfg, cfg.get("seed", 0), device)
safe_lo, safe_hi = map(float, benchmark_data["state_bounds"].detach().cpu().tolist())

def make_pred_plant(method):
    return lambda x, u: sindy_step(sindy, x, u, system=system, method=method)

pred_plants = {m: make_pred_plant(m) for m in ["euler", "rk4"]}

print("Exact safety dynamics: system.perturbed_sindy_model(cfg)")
print("Safety constraints:", [c.name for c in spec.state_constraints])
print("Configured safe box:", safe_lo, safe_hi)

In [ ]:
# Resolve both policies from the uniform checkpoint layout (legacy fallback included).
from sdpc.io import find_nn_checkpoint, find_policy_checkpoint
from sdpc.training import load_nn_policy

sparse_path = find_policy_checkpoint(RESULTS, cfg)
nn_path = find_nn_checkpoint(RESULTS, cfg)
sparse_policy = load_model(sparse_path, device=device).eval()
nn_policy = load_nn_policy(system, nn_path, cfg, device=device)

print("Sparse checkpoint:", sparse_path)
print("NN checkpoint    :", nn_path)
print("Sparse trainable coefficients:", sum(p.numel() for p in sparse_policy.Xi))
print("NN trainable parameters      :", sum(p.numel() for p in nn_policy.parameters()))

In [ ]:
# Build symbolic Jacobian objects once, outside timing.
symbolic_jacobians = {
    method: SymbolicSafetyJacobian(
        sindy, sparse_policy, system.umin, system.umax,
        system=system, action_scale=action_scale, integration_method=method,
    )
    for method in ["euler", "rk4"]
}

xlo, xhi = safe_lo, safe_hi
x0_eval = torch.full((1, system.nx), xhi + 0.10 * (xhi - xlo), dtype=torch.float32, device=device)
r_target = benchmark_data["r"][:, :1, :].contiguous()

CONSTRAINT_MODES = [
    ("barrier", safe_cfg.get("barrier_kind", "squared_hinge")),
    ("boxbarrier", "box_barrier_constraints"),
    ("boxrelu", "box_relu_constraints"),
]
INTEGRATORS = ["euler", "rk4"]
HORIZONS = [5, 10, 20, 40]
WARMUP = 5
N_REP = 20

def make_r_horizon(horizon):
    return r_target.expand(1, horizon, system.nx).contiguous()

def sync():
    if torch.cuda.is_available() and device.type == "cuda":
        torch.cuda.synchronize()

def zero_module_grads(module):
    for p in module.parameters():
        p.grad = None

def safety_loss_for_kind(x_roll, u_roll, kind):
    if kind == "box_relu_constraints":
        loss, _ = rollout_box_relu_loss(x_roll, spec, include_x0=False)
        return loss
    if kind == "box_barrier_constraints":
        loss, _ = rollout_box_barrier_loss(
            x_roll, spec, barrier_kind="squared_hinge", eps=barrier_eps, include_x0=False,
        )
        return loss
    loss, _, _ = rollout_barrier_loss(x_roll, spec, kind=kind, eps=barrier_eps, u_traj=u_roll)
    return loss

def summarize_rows(rows):
    if pd is not None:
        return pd.DataFrame(rows)
    for row in rows:
        print(row)

## 1. Sanity Check Losses

Check the expected behavior.


In [ ]:
h0, method0 = 20, "euler"
r_h = make_r_horizon(h0)
with torch.no_grad():
    xs, us = predict_rollout(
        sparse_policy, pred_plants[method0], x0_eval, r_h, h0,
        umin=system.umin, umax=system.umax, action_scale=action_scale, grad=False,
    )
    xn, un = predict_rollout(
        nn_policy, pred_plants[method0], x0_eval, r_h, h0,
        umin=system.umin, umax=system.umax, action_scale=action_scale, grad=False,
    )
for label, x_roll, u_roll in [("sparse", xs, us), ("nn", xn, un)]:
    vals = {name: float(safety_loss_for_kind(x_roll, u_roll, kind).item()) for name, kind in CONSTRAINT_MODES}
    print(label, vals)

## 2. Gradient-Only Benchmark

Measure safety-gradient runtime.


In [ ]:
def time_sparse_symbolic_gradient_only(horizon, method, kind, n_rep=N_REP, warmup=WARMUP):
    r_h = make_r_horizon(horizon)
    pred = pred_plants[method]
    sj = symbolic_jacobians[method]
    times, losses, unsafe_count = [], [], 0
    for rep in range(warmup + n_rep):
        with torch.no_grad():
            x_roll, u_roll = predict_rollout(
                sparse_policy, pred, x0_eval, r_h, horizon,
                umin=system.umin, umax=system.umax, action_scale=action_scale, grad=False,
            )
            loss = safety_loss_for_kind(x_roll, u_roll, kind)
        unsafe = float(loss.detach().cpu().item()) > safety_loss_tol
        sync(); t0 = time.perf_counter()
        if unsafe:
            sj.rollout_sensitivity_grad(x_roll, u_roll, r_h, spec, kind=kind, eps=barrier_eps)
        sync(); elapsed = time.perf_counter() - t0
        if rep >= warmup:
            times.append(elapsed)
            losses.append(float(loss.detach().cpu().item()))
            unsafe_count += int(unsafe)
    return {"median_s": statistics.median(times), "loss": statistics.median(losses), "unsafe": unsafe_count}

def time_nn_autograd_gradient_only(horizon, method, kind, n_rep=N_REP, warmup=WARMUP):
    r_h = make_r_horizon(horizon)
    pred = pred_plants[method]
    times, losses, unsafe_count = [], [], 0
    for rep in range(warmup + n_rep):
        zero_module_grads(nn_policy)
        x_roll, u_roll = predict_rollout(
            nn_policy, pred, x0_eval, r_h, horizon,
            umin=system.umin, umax=system.umax, action_scale=action_scale, grad=True,
        )
        loss = safety_loss_for_kind(x_roll, u_roll, kind)
        unsafe = float(loss.detach().cpu().item()) > safety_loss_tol
        sync(); t0 = time.perf_counter()
        if unsafe:
            loss.backward()
        sync(); elapsed = time.perf_counter() - t0
        zero_module_grads(nn_policy)
        if rep >= warmup:
            times.append(elapsed)
            losses.append(float(loss.detach().cpu().item()))
            unsafe_count += int(unsafe)
    return {"median_s": statistics.median(times), "loss": statistics.median(losses), "unsafe": unsafe_count}

grad_rows = []
for horizon in HORIZONS:
    for cname, kind in CONSTRAINT_MODES:
        for method in INTEGRATORS:
            sy = time_sparse_symbolic_gradient_only(horizon, method, kind)
            ag = time_nn_autograd_gradient_only(horizon, method, kind)
            grad_rows.append({
                "horizon": horizon,
                "constraint": cname,
                "integrator": method,
                "sparse_symbolic_grad_ms": 1e3 * sy["median_s"],
                "nn_autograd_backward_ms": 1e3 * ag["median_s"],
                "speedup_nn_ag_over_sparse_sym": ag["median_s"] / max(sy["median_s"], 1e-12),
                "sparse_loss": sy["loss"],
                "nn_loss": ag["loss"],
                "sparse_unsafe_reps": sy["unsafe"],
                "nn_unsafe_reps": ag["unsafe"],
            })

summarize_rows(grad_rows)

## 3. Rollout + Gradient Benchmark

Measure rollout and gradient runtime.


In [ ]:
def time_sparse_symbolic_total(horizon, method, kind, n_rep=N_REP, warmup=WARMUP):
    r_h = make_r_horizon(horizon)
    pred = pred_plants[method]
    sj = symbolic_jacobians[method]
    times, losses, unsafe_count = [], [], 0
    for rep in range(warmup + n_rep):
        sync(); t0 = time.perf_counter()
        with torch.no_grad():
            x_roll, u_roll = predict_rollout(
                sparse_policy, pred, x0_eval, r_h, horizon,
                umin=system.umin, umax=system.umax, action_scale=action_scale, grad=False,
            )
            loss = safety_loss_for_kind(x_roll, u_roll, kind)
        unsafe = float(loss.detach().cpu().item()) > safety_loss_tol
        if unsafe:
            sj.rollout_sensitivity_grad(x_roll, u_roll, r_h, spec, kind=kind, eps=barrier_eps)
        sync(); elapsed = time.perf_counter() - t0
        if rep >= warmup:
            times.append(elapsed)
            losses.append(float(loss.detach().cpu().item()))
            unsafe_count += int(unsafe)
    return {"median_s": statistics.median(times), "loss": statistics.median(losses), "unsafe": unsafe_count}

def time_nn_autograd_total(horizon, method, kind, n_rep=N_REP, warmup=WARMUP):
    r_h = make_r_horizon(horizon)
    pred = pred_plants[method]
    times, losses, unsafe_count = [], [], 0
    for rep in range(warmup + n_rep):
        zero_module_grads(nn_policy)
        sync(); t0 = time.perf_counter()
        x_roll, u_roll = predict_rollout(
            nn_policy, pred, x0_eval, r_h, horizon,
            umin=system.umin, umax=system.umax, action_scale=action_scale, grad=True,
        )
        loss = safety_loss_for_kind(x_roll, u_roll, kind)
        unsafe = float(loss.detach().cpu().item()) > safety_loss_tol
        if unsafe:
            loss.backward()
        sync(); elapsed = time.perf_counter() - t0
        zero_module_grads(nn_policy)
        if rep >= warmup:
            times.append(elapsed)
            losses.append(float(loss.detach().cpu().item()))
            unsafe_count += int(unsafe)
    return {"median_s": statistics.median(times), "loss": statistics.median(losses), "unsafe": unsafe_count}

total_rows = []
for horizon in HORIZONS:
    for cname, kind in CONSTRAINT_MODES:
        for method in INTEGRATORS:
            sy = time_sparse_symbolic_total(horizon, method, kind)
            ag = time_nn_autograd_total(horizon, method, kind)
            total_rows.append({
                "horizon": horizon,
                "constraint": cname,
                "integrator": method,
                "sparse_symbolic_total_ms": 1e3 * sy["median_s"],
                "nn_autograd_total_ms": 1e3 * ag["median_s"],
                "speedup_nn_ag_over_sparse_sym": ag["median_s"] / max(sy["median_s"], 1e-12),
                "sparse_loss": sy["loss"],
                "nn_loss": ag["loss"],
                "sparse_unsafe_reps": sy["unsafe"],
                "nn_unsafe_reps": ag["unsafe"],
            })

summarize_rows(total_rows)

## 4. Online Adaptation Trajectory Visualization

Run online policy adaptation.


In [ ]:
from sdpc.plotting import plot_states_and_controls
from sdpc.eval import sample_scenario
from sdpc.adaptation import SafeAdaptationConfig, run_safe_adaptation
from sdpc.adaptation.rollout import clamp_action
import matplotlib.pyplot as plt

# Online-adaptation trajectory comparison on the same sampled scenario.
# Sparse: project safe-adaptation runner with symbolic safety Jacobian.
# NN: local autograd safety-update loop over NN parameters.
visual_method = "rk4"
visual_barrier_kind = "box_barrier_constraints"
visual_cfg = dict(cfg)
visual_cfg["nsteps"] = min(cfg.get("nsteps", 300), 120)
visual_data = sample_scenario(system, visual_cfg, cfg.get("seed", 0), device)
visual_lo, visual_hi = map(float, visual_data["state_bounds"].detach().cpu().tolist())
visual_plant = system.perturbed_plant(visual_cfg)
visual_pred_plant = pred_plants[visual_method]

# The sparse law adapts 6 interpretable coefficients with an LM-normalized update.
# The NN has 1282 weights, so reusing sparse gamma/clip values makes the bounded MLP
# jump into saturated, bang-bang controls. Use conservative global-norm-clipped NN steps.
NN_GAMMA_REF = safe_cfg.get("nn_gamma_ref", 2e-3)
NN_GAMMA_SAFE = safe_cfg.get("nn_gamma_safe", 5e-3)
NN_REF_GRAD_MAX_NORM = safe_cfg.get("nn_ref_grad_max_norm", 1.0)
NN_SAFE_GRAD_MAX_NORM = safe_cfg.get("nn_safe_grad_max_norm", 1.0)
NN_CLIP_UPDATE = safe_cfg.get("nn_clip_update", None)

sparse_online_policy = copy.deepcopy(sparse_policy)
sparse_online_cfg = SafeAdaptationConfig(**{
    **safe_cfg,
    "barrier_kind": visual_barrier_kind,
    "integration_method": visual_method,
    "safety_backend": "symbolic",
})
sparse_online = run_safe_adaptation(
    sparse_online_policy, visual_plant, visual_data, spec, sparse_online_cfg,
    umin=system.umin, umax=system.umax,
    pred_plant=visual_pred_plant,
    derivative_model=sindy,
    system=system,
)

def nn_apply_descent(policy, loss, *, gamma, clip=None, max_norm=None, normalize=False):
    zero_module_grads(policy)
    loss.backward()
    params = [p for p in policy.parameters() if p.requires_grad]
    grads = [(-p.grad.detach().clone() if p.grad is not None else torch.zeros_like(p)) for p in params]
    norm2 = sum((g ** 2).sum() for g in grads)
    raw_norm = torch.sqrt(norm2 + 1e-12)
    scale = raw_norm.new_tensor(1.0)
    if normalize:
        grads = [g / (1.0 + norm2) for g in grads]
    if max_norm is not None:
        scale = torch.clamp(raw_norm.new_tensor(max_norm) / (raw_norm + 1e-12), max=1.0)
        grads = [scale * g for g in grads]
    with torch.no_grad():
        for p, g in zip(params, grads):
            g = torch.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)
            if clip is not None:
                g = g.clamp(min=-clip, max=clip)
            p.add_(gamma * g)
    zero_module_grads(policy)
    return float(raw_norm.detach().cpu().item()), float(scale.detach().cpu().item())

def run_nn_safe_adaptation_autograd(policy, plant, pred_plant, data, spec, cfg_s):
    x_data = data["xn"].detach()
    r_data = data["r"].detach()
    B, _, nx = x_data.shape
    T, nr = r_data.shape[1], r_data.shape[2]
    x0 = x_data[:, 0, :]
    with torch.no_grad():
        u0 = clamp_action(policy(x0, current_reference(r_data, 0)), system.umin, system.umax, cfg_s.action_scale)
    x_traj = torch.empty(B, T, nx, device=device)
    u_traj = torch.empty(B, T, u0.shape[-1], device=device)
    x_traj[:, 0, :] = x0
    u_traj[:, 0, :] = u0
    logs = []

    for t in range(1, T):
        x = x_traj[:, t - 1, :].detach()
        r = current_reference(r_data, t)
        r_h = hold_current_reference(r_data, t, cfg_s.horizon)

        # Reference step: one normalized autograd descent step on one-step tracking loss.
        u_ref = clamp_action(policy(x, r), system.umin, system.umax, cfg_s.action_scale)
        x_ref = pred_plant(x, u_ref)
        ref_loss = 0.5 * ((x_ref - r) ** 2).sum()
        ref_grad_norm, ref_grad_scale = nn_apply_descent(
            policy, ref_loss,
            gamma=NN_GAMMA_REF,
            clip=NN_CLIP_UPDATE,
            max_norm=NN_REF_GRAD_MAX_NORM,
            normalize=True,
        )

        safe_loss = float("inf")
        safe_grad_norm = 0.0
        safe_grad_scale = 1.0
        m = 0
        while True:
            x_roll, u_roll = predict_rollout(
                policy, pred_plant, x, r_h, cfg_s.horizon,
                umin=system.umin, umax=system.umax,
                action_scale=cfg_s.action_scale,
                grad=True,
            )
            loss = safety_loss_for_kind(x_roll, u_roll, cfg_s.barrier_kind)
            safe_loss = float(loss.detach().cpu().item())
            if safe_loss <= cfg_s.safety_loss_tol or m >= cfg_s.max_safety_iters:
                zero_module_grads(policy)
                break
            safe_grad_norm, safe_grad_scale = nn_apply_descent(
                policy, loss,
                gamma=NN_GAMMA_SAFE * cfg_s.safety_gain,
                clip=NN_CLIP_UPDATE,
                max_norm=NN_SAFE_GRAD_MAX_NORM,
                normalize=False,
            )
            m += 1

        with torch.no_grad():
            u_apply = clamp_action(policy(x, r), system.umin, system.umax, cfg_s.action_scale)
            x_next = plant(x, u_apply)
        x_traj[:, t, :] = x_next
        u_traj[:, t, :] = u_apply
        logs.append({
            "t": t, "barrier_loss": safe_loss, "safety_iters": m,
            "ref_grad_norm": ref_grad_norm, "ref_grad_scale": ref_grad_scale,
            "safe_grad_norm": safe_grad_norm, "safe_grad_scale": safe_grad_scale,
        })
    return {"x_traj": x_traj, "u_traj": u_traj, "logs": logs}

nn_online_policy = copy.deepcopy(nn_policy)
nn_online_cfg = SafeAdaptationConfig(**{
    **safe_cfg,
    "barrier_kind": visual_barrier_kind,
    "integration_method": visual_method,
    "safety_backend": "autograd",
})
nn_online = run_nn_safe_adaptation_autograd(
    nn_online_policy, visual_plant, visual_pred_plant, visual_data, spec, nn_online_cfg,
)

box_band = max(float(c.band) for c in spec.state_constraints) if spec.state_constraints else 0.0
print("box-barrier zero-loss state interval:",
      (visual_lo + box_band, visual_hi - box_band),
      "first reference:", visual_data["r"][0, 0].tolist())
print("sparse symbolic total safety iters:", sum(l["safety_iters"] for l in sparse_online["logs"]))
print("NN autograd total safety iters      :", sum(l["safety_iters"] for l in nn_online["logs"]))
print("NN visual gains: gamma_ref=", NN_GAMMA_REF, "gamma_safe=", NN_GAMMA_SAFE,
      "ref_max_norm=", NN_REF_GRAD_MAX_NORM, "safe_max_norm=", NN_SAFE_GRAD_MAX_NORM)

def summarize_online(name, res):
    x = res["x_traj"].detach()
    u = res["u_traj"].detach()
    r = visual_data["r"]
    final_err = torch.linalg.norm(x[:, -1, :] - r[:, -1, :], dim=-1).mean().item()
    print(f"{name:>22s}: final_err={final_err:.4e}, "
          f"xmin={x.min().item():.4f}, xmax={x.max().item():.4f}, "
          f"umin={u.min().item():.4f}, umax={u.max().item():.4f}")

summarize_online("sparse symbolic", sparse_online)
summarize_online("NN autograd", nn_online)

plot_states_and_controls(
    [
        {"x": sparse_online["x_traj"], "u": sparse_online["u_traj"],
         "label": "online sparse symbolic", "color": "green", "linestyle": "-"},
        {"x": nn_online["x_traj"], "u": nn_online["u_traj"],
         "label": "online NN autograd", "color": "darkorange", "linestyle": "--"},
    ],
    r_traj=visual_data["r"],
    xmin=visual_lo,
    xmax=visual_hi,
    state_margin=float(cfg.get("bands", {}).get("box", 0.0)),
    title=f"{SYSTEM}: online safe adaptation, sparse symbolic vs NN autograd ({visual_method})",
)
plt.show()

## 5. Notes

Summarize the comparison.
